In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
for candidate in (
    NOTEBOOK_CWD,
    NOTEBOOK_CWD.parent,
    NOTEBOOK_CWD.parent.parent,
):
    src_dir = candidate / "src"
    if src_dir.exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break

from utils.notebook_env import configure_notebook_environment  # noqa: E402

REPO_ROOT = configure_notebook_environment()


In [ ]:
from utils.notebook_env import resolve_imagecas_base_path
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from utils import (
    load_config_json,
    scale_config_to_resolution,
    run_core_preprocessing_pipeline,
    plot_mip_projection,
    plot_preprocessing_grid,
    plot_stage,
    compute_vesselness_maps,
    plot_vesselness_mip,
    load_raw_img_and_label,
    get_or_detect_aorta_circles,
    get_initial_circle_diagnostics,
    plot_hough_initial_circle,
    plot_hough_refinement_candidates,
    plot_hough_refined_circle,

)

# ==================== Configuracao global dos plots ====================
PLOT_PRESETS = {
    "draft": {
        "dpi": 180,
        "show_titles": True,
        "show_subtitles": True,
        "show_colorbar": True,
        "vesselness_cmap": "gray",
        "invert_label_cmap": False,
    },
    "publication": {
        "dpi": 600,
        "show_titles": False,
        "show_subtitles": False,
        "show_colorbar": False,
        "vesselness_cmap": "gray",
        "invert_label_cmap": True,
    },
}

PLOT_PRESET = "publication"  # "draft" ou "publication"

if PLOT_PRESET not in PLOT_PRESETS:
    raise ValueError(
        f"Preset invalido: {PLOT_PRESET}. Opcoes: {list(PLOT_PRESETS.keys())}"
    )

_plot_cfg = PLOT_PRESETS[PLOT_PRESET]

PLOT_DPI = _plot_cfg["dpi"]
PLOT_SHOW_TITLES = _plot_cfg["show_titles"]
PLOT_SHOW_SUBTITLES = _plot_cfg["show_subtitles"]
PLOT_SHOW_COLORBAR = _plot_cfg["show_colorbar"]
PLOT_VESSELNESS_CMAP = _plot_cfg["vesselness_cmap"]
PLOT_INVERT_LABEL_CMAP = _plot_cfg["invert_label_cmap"]

RUN_VESSELNESS = False


# Análise visual de imagens do ImageCAS

## Carregar os dados

In [ ]:
BASE_PATH = resolve_imagecas_base_path()

CONFIG_FILE = os.path.abspath(os.path.join('..', 'config', 'pipeline_config.json'))
RESOLUTION = 'mid'  # 'high' ou 'mid'

if not os.path.exists(CONFIG_FILE):
    raise FileNotFoundError(f"Arquivo de configuracao nao encontrado: {CONFIG_FILE}")

CONFIG = load_config_json(CONFIG_FILE, {})
if RESOLUTION == 'high':
    CONFIG['DOWNSCALE_FACTORS'] = [1, 1, 1]
CONFIG = scale_config_to_resolution(CONFIG)

print(f"Configuracao carregada de: {CONFIG_FILE}")

ids = np.array([90, 774])
data = {}

for id in ids:
    img_path = f"{BASE_PATH}/{id}.img.nii.gz"
    label_path = f"{BASE_PATH}/{id}.label.nii.gz"
    nii_img, nii_label = load_raw_img_and_label(img_path, label_path)
    img = np.asarray(nii_img.get_fdata(), dtype=np.float32)
    label = np.asarray(nii_label.get_fdata(), dtype=np.uint8)
    spacing = nii_img.header.get_zooms()

    data[id] = {
        "img": img,
        "label": label,
        "spacing": spacing,
    }

print(f"IDs selecionados: {ids.tolist()}")


## MIP das Imagens

In [ ]:
plot_mip_projection(
    data[ids[0]]["img"],
    title="MIP axial - Imagem",
    show_title=PLOT_SHOW_TITLES,
    views=["axial"],
    show_labels=PLOT_SHOW_SUBTITLES,
    dpi=PLOT_DPI,
)

In [ ]:
plot_mip_projection(
    data[ids[1]]["img"],
    title="MIP axial - Imagem",
    show_title=PLOT_SHOW_TITLES,
    views=["axial"],
    show_labels=PLOT_SHOW_SUBTITLES,
    dpi=PLOT_DPI,
)

## MIP dos rótulos

In [ ]:
plot_mip_projection(
    data[ids[0]]["label"],
    title="MIP axial - Rotulo",
    show_title=PLOT_SHOW_TITLES,
    invert_cmap=PLOT_INVERT_LABEL_CMAP,
    views=["axial"],
    show_labels=PLOT_SHOW_SUBTITLES,
    dpi=PLOT_DPI,
)

In [ ]:
plot_mip_projection(
    data[ids[1]]["label"],
    title="MIP axial - Rotulo",
    show_title=PLOT_SHOW_TITLES,
    invert_cmap=PLOT_INVERT_LABEL_CMAP,
    views=["axial"],
    show_labels=PLOT_SHOW_SUBTITLES,
    dpi=PLOT_DPI,
)

## Imagem reduzida, limiarizada e LCC (IDs 90 e 774)


In [ ]:
preprocessed = {}

for img_id in ids:
    down_image, thresh_image, lcc_image, thresh_vals = run_core_preprocessing_pipeline(
        data[img_id]["img"],
        downscale_factors=CONFIG["DOWNSCALE_FACTORS"],
        max_threshold_percentile=CONFIG["MAX_THRESHOLD_PERCENTILE"],
        lcc_per_slice=True,
        use_opencv=False
    )

    center_slice = down_image.shape[2] // 2
    preprocessed[img_id] = {
        "down_image": down_image,
        "thresh_image": thresh_image,
        "lcc_image": lcc_image,
        "thresh_vals": thresh_vals,
        "center_slice": center_slice,
    }

    print(
        f"ID {img_id}: shape={down_image.shape} | fatia central={center_slice}"
    )


In [ ]:
plot_preprocessing_grid(
    preprocessed,
    ids_to_plot=ids,
    mode="slice",
    show_title=PLOT_SHOW_TITLES,
    show_subtitle=PLOT_SHOW_SUBTITLES,
    dpi=PLOT_DPI,
)


In [ ]:
plot_preprocessing_grid(
    preprocessed,
    ids_to_plot=ids,
    mode="mip",
    show_title=PLOT_SHOW_TITLES,
    show_subtitle=PLOT_SHOW_SUBTITLES,
    dpi=PLOT_DPI,
)


### Imagem reduzida - fatia central


In [ ]:
plot_stage(
    preprocessed,
    "down_image",
    "Imagem reduzida",
    img_id=90,
    mode="slice",
    show_title=PLOT_SHOW_TITLES,
    show_subtitle=PLOT_SHOW_SUBTITLES,
    dpi=PLOT_DPI,
)


In [ ]:
plot_stage(
    preprocessed,
    "down_image",
    "Imagem reduzida",
    img_id=774,
    mode="slice",
    show_title=PLOT_SHOW_TITLES,
    show_subtitle=PLOT_SHOW_SUBTITLES,
    dpi=PLOT_DPI,
)


### Imagem reduzida - MIP axial


In [ ]:
plot_stage(
    preprocessed,
    "down_image",
    "Imagem reduzida",
    img_id=90,
    mode="mip",
    show_title=PLOT_SHOW_TITLES,
    show_subtitle=PLOT_SHOW_SUBTITLES,
    dpi=PLOT_DPI,
)


In [ ]:
plot_stage(
    preprocessed,
    "down_image",
    "Imagem reduzida",
    img_id=774,
    mode="mip",
    show_title=PLOT_SHOW_TITLES,
    show_subtitle=PLOT_SHOW_SUBTITLES,
    dpi=PLOT_DPI,
)


### Imagem limiarizada - fatia central


In [ ]:
plot_stage(
    preprocessed,
    "thresh_image",
    "Imagem limiarizada",
    img_id=90,
    mode="slice",
    show_title=PLOT_SHOW_TITLES,
    show_subtitle=PLOT_SHOW_SUBTITLES,
    dpi=PLOT_DPI,
)


In [ ]:
plot_stage(
    preprocessed,
    "thresh_image",
    "Imagem limiarizada",
    img_id=774,
    mode="slice",
    show_title=PLOT_SHOW_TITLES,
    show_subtitle=PLOT_SHOW_SUBTITLES,
    dpi=PLOT_DPI,
)


### Imagem limiarizada - MIP axial


In [ ]:
plot_stage(
    preprocessed,
    "thresh_image",
    "Imagem limiarizada",
    img_id=90,
    mode="mip",
    show_title=PLOT_SHOW_TITLES,
    show_subtitle=PLOT_SHOW_SUBTITLES,
    dpi=PLOT_DPI,
)


In [ ]:
plot_stage(
    preprocessed,
    "thresh_image",
    "Imagem limiarizada",
    img_id=774,
    mode="mip",
    show_title=PLOT_SHOW_TITLES,
    show_subtitle=PLOT_SHOW_SUBTITLES,
    dpi=PLOT_DPI,
)


### Imagem LCC - fatia central


In [ ]:
plot_stage(
    preprocessed,
    "lcc_image",
    "Imagem LCC",
    img_id=90,
    mode="slice",
    show_title=PLOT_SHOW_TITLES,
    show_subtitle=PLOT_SHOW_SUBTITLES,
    dpi=PLOT_DPI,
)


In [ ]:
plot_stage(
    preprocessed,
    "lcc_image",
    "Imagem LCC",
    img_id=774,
    mode="slice",
    show_title=PLOT_SHOW_TITLES,
    show_subtitle=PLOT_SHOW_SUBTITLES,
    dpi=PLOT_DPI,
)


### Imagem LCC - MIP axial


In [ ]:
plot_stage(
    preprocessed,
    "lcc_image",
    "Imagem LCC",
    img_id=90,
    mode="mip",
    show_title=PLOT_SHOW_TITLES,
    show_subtitle=PLOT_SHOW_SUBTITLES,
    dpi=PLOT_DPI,
)


In [ ]:
plot_stage(
    preprocessed,
    "lcc_image",
    "Imagem LCC",
    img_id=774,
    mode="mip",
    show_title=PLOT_SHOW_TITLES,
    show_subtitle=PLOT_SHOW_SUBTITLES,
    dpi=PLOT_DPI,
)


## Transformada de Hough

In [ ]:

hough_diagnostics = {}
hough_detected_circles = {}

for img_id in ids:
    lcc_image = preprocessed[img_id]["lcc_image"]
    first_slice_idx = lcc_image.shape[2] - 1
    spacing = data[img_id]["spacing"]
    dx, dy, dz = (
        spacing[0] * CONFIG["DOWNSCALE_FACTORS"][0],
        spacing[1] * CONFIG["DOWNSCALE_FACTORS"][1],
        spacing[2] * CONFIG["DOWNSCALE_FACTORS"][2],
    )

    radii_start = CONFIG["CIRCLE_DETECTION"]["radii_start_px"]
    radii_end = CONFIG["CIRCLE_DETECTION"]["radii_end_px"]
    radius_step = CONFIG["CIRCLE_DETECTION"]["radius_step_px"]
    hough_radii = np.arange(radii_start, radii_end, radius_step)

    hough_diagnostics[img_id] = get_initial_circle_diagnostics(
        lcc_image[:, :, first_slice_idx],
        hough_radii,
        quadrant_offset=CONFIG["CIRCLE_DETECTION"]["quadrant_offset"],
        total_num_peaks_initial=CONFIG["CIRCLE_DETECTION"]["total_num_peaks_initial"],
        canny_sigma=CONFIG["CIRCLE_DETECTION"]["canny_sigma"],
        neighbor_distance_threshold=CONFIG["CIRCLE_DETECTION"]["neighbor_distance_threshold"],
    )

    hough_detected_circles[img_id] = get_or_detect_aorta_circles(
        img_id,
        lcc_image,
        CONFIG["DOWNSCALE_FACTORS"],
        (dx, dy, dz),
        CONFIG["CIRCLE_DETECTION"],
        base_save_path=None,
        load_cache=False,
        save_cache=False,
    )

### Círculo inicial - ID 90


In [ ]:
plot_hough_initial_circle(
    preprocessed[90]["lcc_image"][:, :, preprocessed[90]["lcc_image"].shape[2] - 1],
    hough_diagnostics[90],
    title="Transformada de Hough - ID 90 | círculo inicial",
    show_title=PLOT_SHOW_TITLES,
    show_subtitle=PLOT_SHOW_SUBTITLES,
    dpi=PLOT_DPI,
 )


In [ ]:
plot_hough_refinement_candidates(
    preprocessed[90]["lcc_image"][:, :, preprocessed[90]["lcc_image"].shape[2] - 1],
    hough_diagnostics[90],
    title="Transformada de Hough - ID 90 | círculos vizinhos para refinamento",
    show_title=PLOT_SHOW_TITLES,
    show_subtitle=PLOT_SHOW_SUBTITLES,
    dpi=PLOT_DPI,
 )


In [ ]:
plot_hough_refined_circle(
    preprocessed[90]["lcc_image"][:, :, preprocessed[90]["lcc_image"].shape[2] - 1],
    hough_diagnostics[90],
    title="Transformada de Hough - ID 90 | círculo final refinado",
    show_title=PLOT_SHOW_TITLES,
    show_subtitle=PLOT_SHOW_SUBTITLES,
    dpi=PLOT_DPI,
 )


### Círculo inicial - ID 774


In [ ]:
plot_hough_initial_circle(
    preprocessed[774]["lcc_image"][:, :, preprocessed[774]["lcc_image"].shape[2] - 1],
    hough_diagnostics[774],
    title="Transformada de Hough - ID 774 | círculo inicial",
    show_title=PLOT_SHOW_TITLES,
    show_subtitle=PLOT_SHOW_SUBTITLES,
    dpi=PLOT_DPI,
 )


In [ ]:
plot_hough_refinement_candidates(
    preprocessed[774]["lcc_image"][:, :, preprocessed[774]["lcc_image"].shape[2] - 1],
    hough_diagnostics[774],
    title="Transformada de Hough - ID 774 | círculos vizinhos para refinamento",
    show_title=PLOT_SHOW_TITLES,
    show_subtitle=PLOT_SHOW_SUBTITLES,
    dpi=PLOT_DPI,
 )


In [ ]:
plot_hough_refined_circle(
    preprocessed[774]["lcc_image"][:, :, preprocessed[774]["lcc_image"].shape[2] - 1],
    hough_diagnostics[774],
    title="Transformada de Hough - ID 774 | círculo final refinado",
    show_title=PLOT_SHOW_TITLES,
    show_subtitle=PLOT_SHOW_SUBTITLES,
    dpi=PLOT_DPI,
 )


## Círculos refinados ao longo do volume


In [ ]:
def generate_circle_slice_figures(
    image, detected_circles, num_samples=6, vmin=None, vmax=None, figsize=(6, 6)
):
    """
    Gera figuras independentes de fatias amostradas com círculos sobrepostos.

    Retorna:
        list: Lista de objetos matplotlib.figure.Figure.
    """
    # Seleciona fatias de amostra onde há círculos detectados.
    slice_indices = sorted(set([c["slice_index"] for c in detected_circles]))

    # Proteção caso a lista venha vazia
    if not slice_indices:
        return []

    step = max(1, len(slice_indices) // num_samples)
    selected_slices = slice_indices[::step][:num_samples]

    figures = []

    for slice_idx in selected_slices:
        circles_in_slice = [
            c for c in detected_circles if c["slice_index"] == slice_idx
        ]

        # Cria uma figura independente para esta fatia.
        fig, ax = plt.subplots(figsize=figsize)

        # Desenha a fatia de fundo.
        ax.imshow(image[:, :, slice_idx], cmap="gray", vmin=vmin, vmax=vmax)

        for circle in circles_in_slice:
            circle_patch = patches.Circle(
                (circle["center_x"], circle["center_y"]),
                circle["radius"],
                fill=False,
                edgecolor="red",
                linewidth=2,
            )
            ax.add_patch(circle_patch)
            ax.plot(circle["center_x"], circle["center_y"], "r+", markersize=10)

        #ax.set_title(f"Fatia {slice_idx}")
        ax.axis("off")

        plt.tight_layout()

        plt.close(fig)

        figures.append(fig)

    return figures

figures_img = {}

for img_id in ids:
    min_val = preprocessed[img_id]["lcc_image"].min()
    max_val = preprocessed[img_id]["lcc_image"].max()
    detected_circles = hough_detected_circles[img_id]

    num_samples = 5

    if len(detected_circles) > num_samples:
        indices = np.linspace(0, len(detected_circles) - 1, num_samples, dtype=int)
        sampled_circles = [detected_circles[i] for i in indices]
    else:
        sampled_circles = detected_circles

    figures_img[img_id] = generate_circle_slice_figures(
        preprocessed[img_id]["lcc_image"], sampled_circles, num_samples=num_samples, vmin=min_val, vmax=max_val
    )

### Círculos refinados espaçados - ID 90


In [ ]:
figures_img[90][0]

In [ ]:
figures_img[90][1]

In [ ]:
figures_img[90][2]

In [ ]:
figures_img[90][3]

### Círculos refinados espaçados - ID 774


In [ ]:
figures_img[774][0]

In [ ]:
figures_img[774][1]

In [ ]:
figures_img[774][2]

In [ ]:
figures_img[774][3]

## MIP dos mapas de vasos


In [ ]:
if RUN_VESSELNESS:
    vessel_maps = compute_vesselness_maps(preprocessed, ids_to_plot=ids)
else:
    vessel_maps = {img_id: None for img_id in ids}


### Óstios

In [ ]:
plot_vesselness_mip(
    vessel_maps,
    img_id=90,
    map_key="vesselness_ostia",
    title="Mapa de vasos para ostios (MIP axial)",
    show_title=PLOT_SHOW_TITLES,
    show_subtitle=PLOT_SHOW_SUBTITLES,
    show_colorbar=PLOT_SHOW_COLORBAR,
    dpi=PLOT_DPI,
    cmap=PLOT_VESSELNESS_CMAP,
)


In [ ]:
plot_vesselness_mip(
    vessel_maps,
    img_id=774,
    map_key="vesselness_ostia",
    title="Mapa de vasos para ostios (MIP axial)",
    show_title=PLOT_SHOW_TITLES,
    show_subtitle=PLOT_SHOW_SUBTITLES,
    show_colorbar=PLOT_SHOW_COLORBAR,
    dpi=PLOT_DPI,
    cmap=PLOT_VESSELNESS_CMAP,
)


### Artérias

In [ ]:
plot_vesselness_mip(
    vessel_maps,
    img_id=90,
    map_key="vesselness_artery",
    title="Mapa de vasos para arterias (MIP axial)",
    show_title=PLOT_SHOW_TITLES,
    show_subtitle=PLOT_SHOW_SUBTITLES,
    show_colorbar=PLOT_SHOW_COLORBAR,
    dpi=PLOT_DPI,
    cmap=PLOT_VESSELNESS_CMAP,
)


In [ ]:
plot_vesselness_mip(
    vessel_maps,
    img_id=774,
    map_key="vesselness_artery",
    title="Mapa de vasos para arterias (MIP axial)",
    show_title=PLOT_SHOW_TITLES,
    show_subtitle=PLOT_SHOW_SUBTITLES,
    show_colorbar=PLOT_SHOW_COLORBAR,
    dpi=PLOT_DPI,
    cmap=PLOT_VESSELNESS_CMAP,
)
